###### Imports and Settings

In [1]:
import pandas as pd
#import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

###### Functions

In [2]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains Decennial SF1 variables.

In [3]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [4]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']

In [5]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828  
**This data guide stops at ID 783 as of 5/18/2022.**

### SF1

In [14]:
dataguide = pd.read_csv('../data guides/DATA GUIDE SF12010.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [15]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]

In [16]:
# ONE 2000
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['SF1 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2010/dec/sf1?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
one = data_appended
#places call
url_str= 'https://api.census.gov/data/2010/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                  |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                  |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                  |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                  |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                  |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
one = one.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2010/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
one = one.append(state, ignore_index = True)
onepull = one
print('Okay Finished')

Okay Finished


In [44]:
one = onepull

In [18]:
# TWO 2000
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['SF1 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2010/dec/sf1?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
two = data_appended
#places call
url_str= 'https://api.census.gov/data/2010/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                  |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                  |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                  |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                  |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                  |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
two = two.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2010/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
two = two.append(state, ignore_index = True)
twopull = two
print('Okay Finished')

Okay Finished


In [45]:
two = twopull

In [46]:
last = two

## Joining

In [47]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
last = last.drop(columns = 'NAME')

In [48]:
data = one.merge(last, on = 'GEO_ID')

In [49]:
transp = data.transpose()
transp.columns = transp.iloc[0]
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Burns town, Tennessee","Hartsville/Trousdale County, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Smyrna town, Tennessee","Slayden town, Tennessee","Dickson city, Tennessee",Tennessee
NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Burns town, Tennessee","Hartsville/Trousdale County, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Smyrna town, Tennessee","Slayden town, Tennessee","Dickson city, Tennessee",Tennessee
GEO_ID,0500000US47161,0500000US47125,0500000US47083,0500000US47085,0500000US47043,0500000US47021,0500000US47147,0500000US47165,0500000US47037,0500000US47189,0500000US47169,0500000US47187,0500000US47149,0500000US47119,1600000US4702180,1600000US4757480,1600000US4759560,1600000US4709880,1600000US4732742,1600000US4739660,1600000US4741200,1600000US4741520,1600000US4744840,1600000US4750780,1600000US4751560,1600000US4752820,1600000US4776860,1600000US4778320,1600000US4778560,1600000US4779980,1600000US4722360,1600000US4728540,1600000US4713080,1600000US4715160,1600000US4769420,1600000US4769080,1600000US4720620,0400000US47
pop,13324,172331,8426,18538,49666,39105,66283,160645,626681,113993,7870,183182,262604,80956,4541,2093,4149,1468,7870,2756,32588,26190,1750,23671,108755,1951,395,1477,4105,3206,604,30278,1235,132929,39974,178,14538,6346105
agebysex_total_series,13324,172331,8426,18538,49666,39105,66283,160645,626681,113993,7870,183182,262604,80956,4541,2093,4149,1468,7870,2756,32588,26190,1750,23671,108755,1951,395,1477,4105,3206,604,30278,1235,132929,39974,178,14538,6346105
age_total_male,6655,84454,4144,9115,24384,19535,32634,78389,303540,55834,3912,89336,129726,39155,2208,1026,2027,708,3912,1384,16006,12502,824,11363,53422,952,204,698,1904,1568,280,14534,685,64768,19461,86,6757,3093504


In [50]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [51]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [52]:
data = transp

In [53]:
GNRCCounties = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee']]
data['GNRC'] = sum(GNRCCounties)

In [54]:
MPOCounties = [data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],
               data['Williamson County, Tennessee'],data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['MPO'] = sum(MPOCounties)

In [55]:
RuthInc = [data['Eagleville city, Tennessee'],data['La Vergne city, Tennessee'],data['Murfreesboro city, Tennessee'],data['Smyrna town, Tennessee']]
data['Rutherford Incorporated'] = sum(RuthInc)
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']
WilsonInc = [data['Lebanon city, Tennessee'],data['Mount Juliet city, Tennessee'],data['Watertown city, Tennessee']]
data['Wilson Incorporated'] = sum(WilsonInc)
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']
CheathInc = [data['Ashland City town, Tennessee'],data['Kingston Springs town, Tennessee'],data['Pegram town, Tennessee'],data['Pleasant View city, Tennessee']]
data['Cheatham Incorporated'] = sum(CheathInc)
data['Cheatham Unincorporated'] = data['Cheatham County, Tennessee'] - data['Cheatham Incorporated']
DicksInc = [data['Burns town, Tennessee'],data['Charlotte town, Tennessee'],data['Dickson city, Tennessee'],data['Slayden town, Tennessee'],
            data['Vanleer town, Tennessee'],data['White Bluff town, Tennessee']]
data['Dickson Incorporated'] = sum(DicksInc)
data['Dickson Unincorporated'] = data['Dickson County, Tennessee'] - data['Dickson Incorporated']
HumphInc = [data['McEwen city, Tennessee'],data['New Johnsonville city, Tennessee'],data['Waverly city, Tennessee']]
data['Humphreys Incorporated'] = sum(HumphInc)
data['Humphreys Unincorporated'] = data['Humphreys County, Tennessee'] - data['Humphreys Incorporated']
data['Montgomery Incorporated'] = data['Clarksville city, Tennessee']
data['Montgomery Unincorporated'] = data['Montgomery County, Tennessee'] - data['Montgomery Incorporated']
# data['Trousdale Incorporated'] = data['Hartsville/Trousdale County, Tennessee']
# data['Trousdale Unincorporated'] = data['Trousdale County, Tennessee'] - data['Trousdale Incorporated']

In [56]:
data = data.transpose().reset_index()

In [57]:
data.head()

,NAME,pop,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_total_series,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits_mortgage,tenure_owneroccunits_nomortgage,tenure_renteroccunits,hhtype_total_series,hhtype_familyhh,hhtype_familyhh_marriedcouplefam,hhtype_familyhh_otherfam,hhtype_familyhh_malenospouse,hhtype_familyhh_femalenospouse,hhtype_nonfamhh,hhtype_nonfamhh_householderalone,hhtype_nonfamhh_householdernotalone,StateFIPS,GeoFIPS
0,"Stewart County, Tennessee",13324.0,13324.0,6655.0,358.0,462.0,464.0,291.0,184.0,72.0,78.0,191.0,317.0,362.0,371.0,469.0,568.0,495.0,485.0,168.0,292.0,174.0,221.0,263.0,187.0,103.0,80.0,6669.0,331.0,390.0,456.0,294.0,159.0,53.0,63.0,202.0,320.0,325.0,425.0,453.0,526.0,532.0,493.0,178.0,274.0,173.0,216.0,292.0,223.0,150.0,141.0,13324.0,12605.0,188.0,75.0,137.0,3.0,81.0,235.0,12469.0,250.0,2.46,6778.0,6778.0,5386.0,1392.0,5386.0,2432.0,1726.0,1228.0,5386.0,3793.0,2995.0,798.0,281.0,517.0,1593.0,1373.0,220.0,47.0,161.0
1,"Montgomery County, Tennessee",172331.0,172331.0,84454.0,7803.0,6754.0,6257.0,3697.0,2498.0,1435.0,1507.0,5140.0,8306.0,6592.0,5883.0,5495.0,5379.0,4839.0,3804.0,1367.0,1746.0,909.0,1202.0,1563.0,1195.0,679.0,404.0,87877.0,7526.0,6548.0,6027.0,3602.0,2756.0,1583.0,1545.0,4994.0,8074.0,6722.0,6281.0,5630.0,5778.0,5310.0,4198.0,1505.0,1959.0,1062.0,1441.0,2004.0,1489.0,936.0,907.0,172331.0,122336.0,32982.0,988.0,3570.0,676.0,4164.0,7615.0,115553.0,13752.0,2.65,70098.0,70098.0,63673.0,6425.0,63673.0,30811.0,8695.0,24167.0,63673.0,45596.0,33143.0,12453.0,2914.0,9539.0,18077.0,14302.0,3775.0,47.0,125.0
2,"Houston County, Tennessee",8426.0,8426.0,4144.0,245.0,304.0,287.0,186.0,124.0,48.0,45.0,113.0,214.0,228.0,242.0,264.0,282.0,317.0,295.0,104.0,169.0,110.0,140.0,180.0,117.0,86.0,44.0,4282.0,228.0,276.0,298.0,168.0,92.0,45.0,45.0,133.0,202.0,223.0,268.0,276.0,307.0,281.0,329.0,120.0,165.0,107.0,141.0,192.0,162.0,107.0,117.0,8426.0,8015.0,193.0,24.0,25.0,3.0,35.0,131.0,7931.0,129.0,2.46,4188.0,4188.0,3349.0,839.0,3349.0,1365.0,1179.0,805.0,3349.0,2285.0,1752.0,533.0,165.0,368.0,1064.0,942.0,122.0,47.0,83.0
3,"Humphreys County, Tennessee",18538.0,18538.0,9115.0,539.0,569.0,627.0,460.0,266.0,92.0,106.0,263.0,433.0,532.0,592.0,579.0,665.0,682.0,639.0,273.0,345.0,211.0,345.0,378.0,252.0,172.0,95.0,9423.0,502.0,613.0,596.0,387.0,213.0,99.0,77.0,260.0,471.0,521.0,595.0,602.0,681.0,693.0,720.0,261.0,380.0,217.0,308.0,424.0,331.0,250.0,222.0,18538.0,17629.0,463.0,82.0,35.0,6.0,101.0,222.0,17498.0,278.0,2.46,8865.0,8865.0,7454.0,1411.0,7454.0,3062.0,2561.0,1831.0,7454.0,5116.0,3887.0,1229.0,369.0,860.0,2338.0,1969.0,369.0,47.0,85.0
4,"Dickson County, Tennessee",49666.0,49666.0,24384.0,1686.0,1797.0,1829.0,1173.0,619.0,295.0,278.0,839.0,1462.0,1503.0,1633.0,1725.0,1926.0,1870.0,1510.0,585.0,794.0,465.0,603.0,770.0,514.0,286.0,222.0,25282.0,1602.0,1645.0,1659.0,1046.0,560.0,278.0,256.0,844.0,1537.0,1529.0,1733.0,1854.0,1963.0,1962.0,1653.0,594.0,842.0,519.0,640.0,885.0,665.0,495.0,521.0,49666.0,45611.0,2056.0,172.0,226.0,16.0,684.0,901.0,44880.0,1573.0,2.57,20820

# Come back for hartsville/trousdale.... not there for some reason

# Calculations

## Population + Projections Summary: 

In [58]:
data['Population'] = data['agebysex_total_series']
data = data.drop(columns = ['agebysex_total_series','pop'])

## Age Sex Race Summary:

### Age Brackets
+ M and F U5, 5 to 9, 10 to 14, 15 to 17, 18 to 24, 25 to 34, 35 to 44, 45 to 54, 55 to 64, 65 to 74, 75 to 84, 85+  
+ age brackets: under 18, 18 to 54, 55+  
+ 65+ 

In [59]:
#small groups m and f
data['Male Under 5'] = data['age_m_u5']
data['Female Under 5'] = data['age_f_u5']
data['Male 5 to 9'] = data['age_m_5to9']
data['Female 5 to 9'] = data['age_f_5to9']
data['Male 10 to 14'] = data['age_m_10to14']
data['Female 10 to 14'] = data['age_f_10to14']
data['Male 15 to 17'] = data['age_m_15to17']
data['Female 15 to 17'] = data['age_f_15to17']
data['Male 18 to 24'] = data['age_m_18to19']+data['age_m_20']+data['age_m_21']+data['age_m_22to24']
data['Female 18 to 24'] = data['age_f_18to19']+data['age_f_20']+data['age_f_21']+data['age_f_22to24']
data['Male 25 to 34'] = data['age_m_25to29']+data['age_m_30to34']
data['Female 25 to 34'] = data['age_f_25to29']+data['age_f_30to34']
data['Male 35 to 44'] = data['age_m_35to39']+data['age_m_40to44']
data['Female 35 to 44'] = data['age_f_35to39']+data['age_f_40to44']
data['Male 45 to 54'] = data['age_m_45to49']+data['age_m_50to54']
data['Female 45 to 54'] = data['age_f_45to49']+data['age_f_50to54']
data['Male 55 to 64'] = data['age_m_55to59']+data['age_m_60to61']+data['age_m_62to64']
data['Female 55 to 64'] = data['age_f_55to59']+data['age_f_60to61']+data['age_f_62to64']
data['Male 65 to 74'] = data['age_m_65to66']+data['age_m_67to69']+data['age_m_70to74']
data['Female 65 to 74'] = data['age_f_65to66']+data['age_f_67to69']+data['age_f_70to74']
data['Male 75 to 84'] = data['age_m_75to79']+data['age_m_80to84']
data['Female 75 to 84'] = data['age_f_75to79']+data['age_f_80to84']
data['Male 85 and Older'] = data['age_m_85+']
data['Female 85 and Older'] = data['age_f_85+']

#age brackets
u18list = [data['Male Under 5'],data['Female Under 5'],data['Male 5 to 9'],data['Female 5 to 9'],data['Male 10 to 14'],data['Female 10 to 14'],data['Male 15 to 17'],
           data['Female 15 to 17']]
data['Age:Under 18'] = sum(u18list)
eighteento54list = [data['Male 18 to 24'],data['Female 18 to 24'],data['Male 25 to 34'],data['Female 25 to 34'],data['Male 35 to 44'],data['Female 35 to 44'],
              data['Male 45 to 54'],data['Female 45 to 54']]
data['Age:18 to 24'] = sum(eighteento54list)
fifty5pluslist = [data['Male 55 to 64'],data['Female 55 to 64'],data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],
                  data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:55 and Older'] = sum(fifty5pluslist)

#65+
sixty5pluslist = [data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],data['Male 85 and Older'],data['Female 85 and Older']]
data['Age:65 and Older'] = sum(sixty5pluslist)

cols = ['age_total_male','age_total_female','age_m_u5','age_m_5to9','age_m_10to14','age_m_15to17','age_m_18to19','age_m_20','age_m_21','age_m_22to24','age_m_25to29','age_m_30to34','age_m_35to39',
        'age_m_40to44','age_m_45to49','age_m_50to54','age_m_55to59','age_m_60to61','age_m_62to64','age_m_65to66','age_m_67to69','age_m_70to74','age_m_75to79',
        'age_m_80to84','age_m_85+','age_f_u5','age_f_5to9','age_f_10to14','age_f_15to17','age_f_18to19','age_f_20','age_f_21','age_f_22to24','age_f_25to29',
        'age_f_30to34','age_f_35to39','age_f_40to44','age_f_45to49','age_f_50to54','age_f_55to59','age_f_60to61','age_f_62to64','age_f_65to66','age_f_67to69',
        'age_f_70to74','age_f_75to79','age_f_80to84','age_f_85+']
data = data.drop(columns = cols)

### Race/Ethnicity #s   
+ White Alone  
+ Black/African American Alone  
+ American Indian Alaska Native Alone  
+ Asian Alone  
+ Native Hawaiian/Other Pacific Islander Alone  
+ Some Other Race Alone  
+ Two or More Races  
+ Hispanic/Latino  
+ White, Not Hispanic/Latino  
+ Total Minority (non-white, non Hispanic/Latino)

In [60]:
#White Alone
data['White Alone'] = data['raceeth_white_alone']
data['White Alone %'] = data['White Alone']/data['Population']
#Black or African American Alone 
data['Black or African American Alone'] = data['raceeth_blackafricanamerican_alone']
data['Black or African American Alone %'] = data['Black or African American Alone']/data['Population']
#American Indian and Alaska Native Alone
data['American Indian Alaska Native Alone'] = data['raceeth_americanindianalaskanative_alone']
data['American Indian Alaska Native Alone %'] = data['American Indian Alaska Native Alone']/data['Population']
#Asian Alone
data['Asian Alone'] = data['raceeth_asian_alone']
data['Asian Alone %'] = data['Asian Alone']/data['Population']
#Native Hawaiian Other Pacific Islander Alone
data['Native Hawaiian Other Pacific Islander Alone'] = data['raceeth_nativehawaiianotherpacificislander_alone']
data['Native Hawaiian Other Pacific Islander Alone %'] = data['Native Hawaiian Other Pacific Islander Alone']/data['Population']
#Some Other Race Alone
data['Some Other Race Alone'] = data['raceeth_someotherrace_alone']
data['Some Other Race Alone %'] = data['Some Other Race Alone']/data['Population']
#Two or More Races
data['Two or More Races'] = data['raceeth_twoormoreraces_alone']
data['Two or More Races %'] = data['Two or More Races']/data['Population']
#Hispanic or Latino
data['Hispanic or Latino'] = data['raceeth_hispanicorlatino']
data['Hispanic or Latino %'] = data['Hispanic or Latino']/data['Population']
#Data Minority
data['Minority'] = data['Population'] - data['raceeth_whitealone_nothispanicorlatino']
data['Minority %'] = data['Minority']/data['Population']
cols = ['raceeth_total_series','raceeth_white_alone','raceeth_blackafricanamerican_alone','raceeth_americanindianalaskanative_alone','raceeth_asian_alone',
        'raceeth_nativehawaiianotherpacificislander_alone','raceeth_someotherrace_alone','raceeth_twoormoreraces_alone','raceeth_hispanicorlatino',
        'raceeth_whitealone_nothispanicorlatino']
data = data.drop(columns = cols)

## Household Summary:  

### Households by Household Type  
+ Total Households  
+ Family Households  
+ Family Households: Married-Couple Family  
+ Family Households: Other Family  
+ Family Households: Other Family: Male Householder no Wife Present  
+ Family Households: Other Family: Female Householder no Husband Present  
+ Nonfamily Households  
+ Nonfamily Household: Male Householder  
+ Nonfamily Household: Female Householder  

*They don't have the nonfamily male or female - replaced by Householder alone or not alone*

In [61]:
data['Total Households'] = data['hhtype_total_series']
data['Family Households'] = data['hhtype_familyhh']
data['Family Households: Married Couple Family'] = data['hhtype_familyhh_marriedcouplefam']
data['Family Households: Not Married Couple Family'] = data['hhtype_familyhh_otherfam']
data['Family Households: Not Married Couple: Male no Spouse'] = data['hhtype_familyhh_malenospouse']
data['Family Households: Not Married Couple: Female no Spouse'] = data['hhtype_familyhh_femalenospouse']
data['Nonfamily Households'] = data['hhtype_nonfamhh']
data['Nonfamily Households: Householder Alone'] = data['hhtype_nonfamhh_householderalone']
data['Nonfamily Households: Householder not Alone'] = data['hhtype_nonfamhh_householdernotalone']

cols = ['hhtype_total_series','hhtype_familyhh','hhtype_familyhh_marriedcouplefam','hhtype_familyhh_otherfam','hhtype_familyhh_malenospouse',
        'hhtype_familyhh_femalenospouse','hhtype_nonfamhh','hhtype_nonfamhh_householderalone','hhtype_nonfamhh_householdernotalone']
data = data.drop(columns = cols)

### Average Household Size 
Median - drop for incorporated and unincorporated

In [62]:
data['Average Household Size'] = data['hhsize_avg']
data = data.drop(columns = ['hhsize_avg'])

### Occupancy Status, and Percentages  
+ Occupancy Total Households
+ Occupied  
+ Vacant  

In [63]:
data['Occupancy:Total Households'] = data['occupancy_total_series']
data['Occupancy:Occupied Units'] = data['occupancy_occupiedunits']
data['Occupancy%:Occupied Units'] = percent(data['Occupancy:Occupied Units'], data['Occupancy:Total Households'])
data['Occupancy:Vacant Units'] = data['occupancy_vacantunits']
data['Occupancy%:Vacant Units'] = percent(data['Occupancy:Vacant Units'], data['Occupancy:Total Households'])

cols = ['occupancy_total_series','occupancy_occupiedunits','occupancy_vacantunits']
data = data.drop(columns = cols)

### Tenure, and Percentages  
+ Tenure Total Households  
+ Owners  
+ Renters

In [64]:
data['Tenure:Total Households'] = data['tenure_total_series']
data['Tenure:Owners'] = data['tenure_owneroccunits_mortgage']+data['tenure_owneroccunits_nomortgage']
data['Tenure%:Owners'] = percent(data['Tenure:Owners'], data['Tenure:Total Households'])
data['Tenure:Renters'] = data['tenure_renteroccunits']
data['Tenure%:Renters'] = percent(data['Tenure:Renters'], data['Tenure:Total Households'])
cols = ['units_allhousing','tenure_total_series','tenure_owneroccunits_mortgage','tenure_owneroccunits_nomortgage','tenure_renteroccunits']
data = data.drop(columns = cols)

In [65]:
data.head()

,NAME,StateFIPS,GeoFIPS,Population,Male Under 5,Female Under 5,Male 5 to 9,Female 5 to 9,Male 10 to 14,Female 10 to 14,Male 15 to 17,Female 15 to 17,Male 18 to 24,Female 18 to 24,Male 25 to 34,Female 25 to 34,Male 35 to 44,Female 35 to 44,Male 45 to 54,Female 45 to 54,Male 55 to 64,Female 55 to 64,Male 65 to 74,Female 65 to 74,Male 75 to 84,Female 75 to 84,Male 85 and Older,Female 85 and Older,Age:Under 18,Age:18 to 24,Age:55 and Older,Age:65 and Older,White Alone,White Alone %,Black or African American Alone,Black or African American Alone %,American Indian Alaska Native Alone,American Indian Alaska Native Alone %,Asian Alone,Asian Alone %,Native Hawaiian Other Pacific Islander Alone,Native Hawaiian Other Pacific Islander Alone %,Some Other Race Alone,Some Other Race Alone %,Two or More Races,Two or More Races %,Hispanic or Latino,Hispanic or Latino %,Minority,Minority %,Total Households,Family Households,Family Households: Married Couple Family,Family Households: Not Married Couple Family,Family Households: Not Married Couple: Male no Spouse,Family Households: Not Married Couple: Female no Spouse,Nonfamily Households,Nonfamily Households: Householder Alone,Nonfamily Households: Householder not Alone,Average Household Size,Occupancy:Total Households,Occupancy:Occupied Units,Occupancy%:Occupied Units,Occupancy:Vacant Units,Occupancy%:Vacant Units,Tenure:Total Households,Tenure:Owners,Tenure%:Owners,Tenure:Renters,Tenure%:Renters
0,"Stewart County, Tennessee",47.0,161.0,13324.0,358.0,331.0,462.0,390.0,464.0,456.0,291.0,294.0,525.0,477.0,679.0,645.0,840.0,878.0,1063.0,1058.0,945.0,945.0,658.0,681.0,290.0,373.0,80.0,141.0,3046.0,6165.0,4113.0,2223.0,12605.0,0.946037,188.0,0.014110,75.0,0.005629,137.0,0.010282,3.0,0.000225,81.0,0.006079,235.0,0.017637,250.0,0.018763,855.0,0.064170,5386.0,3793.0,2995.0,798.0,281.0,517.0,1593.0,1373.0,220.0,2.46,6778.0,5386.0,79.46,1392.0,20.54,5386.0,4158.0,77.20,1228.0,22.80
1,"Montgomery County, Tennessee",47.0,125.0,172331.0,7803.0,7526.0,6754.0,6548.0,6257.0,6027.0,3697.0,3602.0,10580.0,10878.0,14898.0,14796.0,11378.0,11911.0,10218.0,11088.0,6917.0,7662.0,3674.0,4507.0,1874.0,2425.0,404.0,907.0,48214.0,95747.0,28370.0,13791.0,122336.0,0.709890,32982.0,0.191388,988.0,0.005733,3570.0,0.020716,676.0,0.003923,4164.0,0.024163,7615.0,0.044188,13752.0,0.079800,56778.0,0.329471,63673.0,45596.0,33143.0,12453.0,2914.0,9539.0,18077.0,14302.0,3775.0,2.65,70098.0,63673.0,90.83,6425.0,9.17,63673.0,39506.0,62.05,24167.0,37.95
2,"Houston County, Tennessee",47.0,83.0,8426.0,245.0,228.0,304.0,276.0,287.0,298.0,186.0,168.0,330.0,315.0,442.0,425.0,506.0,544.0,599.0,588.0,568.0,614.0,430.0,440.0,203.0,269.0,44.0,117.0,1992.0,3749.0,2685.0,1503.0,8015.0,0.951222,193.0,0.022905,24.0,0.002848,25.0,0.002967,3.0,0.000356,35.0,0.004154,131.0,0.015547,129.0,0.015310,495.0,0.058747,3349.0,2285.0,1752.0,533.0,165.0,368.0,1064.0,942.0,122.0,2.46,4188.0,3349.0,79.97,839.0,20.03,3349.0,2544.0,75.96,805.0,24.04
3,"Humphreys County, Tennessee",47.0,85.0,18538.0,539.0,502.0,569.0,613.0,627.0,596.0,460.0,387.0,727.0,649.0,965.0,992.0,1171.0,1197.0,1347.0,1374.0,1257.0,1361.0,934.0,949.0,424.0,581.0,95.0,222.0,4293.0,8422.0,5823.0,3205.0,17629.0,0.950966,463.0,0.024976,82.0,0.004423,35.0,0.001888,6.0,0.000324,101.0,0.005448,222.0,0.011975,278.0,0.014996,1040.0,0.056101,7454.0,5116.0,3887.0,1229.0,369.0,860.0,2338.0,1969.0,369.0,2.46,8865.0,7454.0,84.08,1411.0,15.92,7454.0,5623.0,75.44,1831.0,24.56
4,"Dickson County, Tennessee",47.0,43.0,49666.0,1686.0,1602.0,1797.0,1645.0,1829.0,1659.0,1173.0,1046.0,2031.0,1938.0,2965.0,3066.0,3358.0,3587.0,3796.0,3925.0,2889.0,3089.0,1838.0,2044.0,800.0,1160.0,222.0,521.0,12437.0,24666.0,12563.0,6585.0,45611.0,0.918355,2056.0,0.041397,172.0,0.003463,226.0,0.004550,16.0,0.000322,684.0,0.013772,901.0,0.018141,1573.0,0.031672,4786.0,0.096364,19107.0,13598.0,10122.0,3476.0,974.0,2502.0,5509.0,4574.0,935.0,2.57,20820.0,19107.0,91.77,1713.0,8.23,19107.0,13984.0,73.19,5123.0,26.81
